# 04 - LLM e relatorios

Neste notebook, utilizamos a solução final obtida anteriormente para gerar artefatos textuais de apoio operacional. O foco desta etapa e transformar os resultados do algoritmo genético em instrucoes e relatorios mais interpretaveis para equipes humanas.

## 1. Carregamento da amostra oficial e da frota final

Nesta etapa, carregamos a amostra final de entregas e a frota recalibrada para reconstruir a solução final que servira de base para os textos operacionais e para o prompt da LLM.

In [1]:
from pathlib import Path
import pandas as pd
import sys

BASE_DIR = Path("..").resolve()
SAMPLES_DIR = BASE_DIR / "dataset" / "samples"
SRC_DIR = BASE_DIR / "src"
RESULTS_DIR = BASE_DIR / "results"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from médical_route_optimizer import (
    build_driver_instructions,
    build_llm_prompt,
    build_operations_report,
    load_stops,
    load_vehicles,
    optimize_routes,
)

stops_path = SAMPLES_DIR / "stops_sample_100.csv"
vehicles_path = SAMPLES_DIR / "vehicles_sample_100_experiment.csv"

stops_df = pd.read_csv(stops_path)
vehicles_df = pd.read_csv(vehicles_path)

stops = load_stops(stops_path)
vehicles = load_vehicles(vehicles_path)

print("Stops shape:", stops_df.shape)
print("Vehicles shape:", vehicles_df.shape)

Stops shape: (101, 7)
Vehicles shape: (4, 3)


## 2. Reconstrucao da solução final

Para manter consistencia com o Notebook 3, reconstruimos a solução final usando a configuração selecionada como melhor experimento. Essa solução será usada como base para os textos operacionais.

In [2]:
final_config = {
    "population_size": 120,
    "generations": 120,
    "mutation_rate": 0.10,
    "elite_size": 2,
    "random_seed": 42,
}

final_result = optimize_routes(
    stops=stops,
    vehicles=vehicles,
    population_size=final_config["population_size"],
    generations=final_config["generations"],
    mutation_rate=final_config["mutation_rate"],
    elite_size=final_config["elite_size"],
    random_seed=final_config["random_seed"],
)

print("Best fitness:", final_result.best_fitness)
print("Total distance:", final_result.total_distance)

Best fitness: 31519.0
Total distance: 605.7


## 3. Geracao dos artefatos textuais

Nesta etapa, produzimos tres saidas principais: as instrucoes operacionais para os motoristas, o relatorio de apoio gerencial e o prompt-base para integração com uma LLM. Esses artefatos apróximam a solução técnica do uso humano e interpretavel.

In [ ]:
driver_instructions = build_driver_instructions(final_result, stops)
operations_report = build_operations_report(final_result, stops, vehicles)
llm_prompt = build_llm_prompt(final_result, stops, vehicles)

print("Instrucoes para motoristas:")
print(driver_instructions[:1500])

print("\n" + "=" * 80 + "\n")

print("Relatorio operacional:")
print(operations_report[:1500])

print("\n" + "=" * 80 + "\n")

print("Prompt para LLM:")
print(llm_prompt[:1500])

## 4. Persistencia das saidas textuais

Depois da geracao, salvamos os artefatos textuais em arquivos para reutilizacao no relatorio técnico, no vídeo e em testes de integração com modelos de linguagem.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

driver_instructions_path = RESULTS_DIR / "driver_instructions_final.md"
operations_report_path = RESULTS_DIR / "operations_report_final.md"
llm_prompt_path = RESULTS_DIR / "llm_prompt_final.txt"

driver_instructions_path.write_text(driver_instructions, encoding="utf-8")
operations_report_path.write_text(operations_report, encoding="utf-8")
llm_prompt_path.write_text(llm_prompt, encoding="utf-8")

print("Instrucoes salvas em:", driver_instructions_path)
print("Relatorio salvo em:", operations_report_path)
print("Prompt salvo em:", llm_prompt_path)

## 5. Sintese da etapa

A partir da solução final obtida anteriormente, foram gerados tres artefatos textuais de apoio: instrucoes operacionais para motoristas, relatorio gerencial e prompt-base para integração com LLM. Esses materiais permitem traduzir a saida numérica do algoritmo genético em formatos mais interpretaveis para uso humano.

Embora os textos ainda reflitam a abstração adotada no mapeamento das categorias da base, a estrutura final cumpre o objetivo de demonstrar como a solução pode apoiar a comunicacao operacional e a apresentacao de resultados no contexto do projeto.